# DS6 — Stochastic Processes (StoP) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS6 (Danish Adolescents, quasi-experimental, N=243 behavioral)
**Model family:** Stochastic Differential Equations (StoP)
**Date:** 2026-08-06
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds6_stop_blueprint.md`

---

## Scope and purpose

First DS6 CAS model. Establishes DS6's own state variable (daily attempt count, split by
`study_period`) and drift/diffusion estimates other DS6 families (InT, DyS) will build on.

- **Goal:** Fit a count-based stochastic process to DS6's daily app-opening volume,
  separately for the baseline and intervention periods, and compare drift/diffusion/
  burstiness within participants across that period boundary.
- **Not goal:** Phenotype classification, personalization, cross-dataset harmonization.
- **Output:** Per-participant, per-period SDE parameters (mu_hat, sig_hat, Fano_factor),
  a within-participant period-effect quantity (h_period), condition x period summary,
  and preliminary psychosocial correlates.

### FLAG decisions in effect (blueprint Part 6)

| Flag | Decision |
|---|---|
| FLAG 1 | Constant-drift process on zero-filled daily counts (not OU, not a point-process intensity model) |
| FLAG 2 | Method-of-moments estimators (mean/std of increments) |
| FLAG 3 | Period effect as drift difference: h_period = mu_hat_intervention - mu_hat_baseline (within-participant) |
| FLAG 4 | MU_THRESHOLD = 0.5 attempts/day (rescaled from DS5's 0.005, which was tuned for a [0,1] rate) |
| FLAG 5 | StM/InT Boltzmann-type bridges: forward connections only, not implemented here |

### WARNING — proxy status of the state variable, and why this is NOT a DS5 port

**DS6 has no dismiss/close outcomes — every row has `event_outcome_simple == 'opened'`.**
DS5's state variable `X_t = n_opened/n_events` would be identically 1.0 for every DS6
participant every day: zero information. DS5's formula does not transfer.

The theory paper's own Section 1.3 names the fallback used here by name: *"...event logs can
be aggregated into user-day or user-week time series, such as **daily attempt count**, daily
dismissal rate, daily opening rate, or daily friction-success rate."* This notebook uses daily
attempt count — the one option of the four that does not require an outcome variable DS6
does not have.

**N_t^(i,period) = daily count of app-opening events for participant i, within a given
`study_period`.** This is a count-based behavioral volume measurement. It approximates
attempt frequency and its day-to-day trend/variability. It does NOT directly measure urge,
self-control capacity, habit strength, or any latent cognitive/psychological state. This
distinction must be maintained throughout interpretation.

---

## Section 1: Pure Theory — Stochastic Processes in CAS

### What StoP models capture (paper Section 2.3 / 1.3)

A Stochastic Differential Equation (SDE) models a system evolving under deterministic drift
plus random diffusion. Appropriate when: the system has a continuous (or count) state that
evolves over time, there is a discernible trend (drift), and there is genuine day-to-day
randomness beyond measurement error.

### The core SDE (general form, same equation family as DS5/Structured)

    dN_t = mu * dt + sigma * dW_t

where `N_t` = daily attempt count (this notebook's state variable, not DS5's [0,1] rate),
`mu` = drift (attempts/day trend), `sigma` = diffusion (day-to-day variability), `W_t` =
standard Wiener process.

### Discrete-time approximation: Euler-Maruyama-style increments

With time step delta_t = 1 day:

    N_{n+1} = N_n + mu + sigma * Z_n,   Z_n ~ N(0,1) i.i.d.

Unlike DS5, values are NOT clipped to [0,1] — `N_t` is an unbounded non-negative count.
(Simulation/forward-projection is out of scope for this notebook; only estimation is
performed here — see blueprint Part 6, Boltzmann/simulation bridges deferred.)

### Fano factor (count-process-specific; has no DS5 analog)

    Fano_factor = Var(N_t) / Mean(N_t)     computed on the raw (non-differenced) daily grid

`Fano ~= 1`: Poisson-like, evenly random. `Fano > 1`: overdispersed/bursty. `Fano < 1`: more
regular than chance. Not meaningful for DS5's bounded proportion; a real quantity here because
`N_t` is an unbounded count, same role it played in the Structured StoP notebook.

### Period effect (DS6-specific re-derivation of DS5's condition-drift-reduction logic)

    h_period^(i) = mu_hat^(i,intervention) - mu_hat^(i,baseline)

DS5's `h_c` compared drift *between conditions* (a between-subject RCT arm comparison). DS6's
`h_period` compares drift *within the same participant, before vs after* — a genuinely
different, arguably stronger comparison, made possible only because DS6 logs an explicit
baseline period in every row (see blueprint Part 1).

In [ ]:
# =============================================================================
# Imports and paths
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from datetime import datetime

ROOT    = Path('..').resolve()
CLEANED = ROOT / 'outputs' / 'cleaned'
FEAT    = ROOT / 'outputs' / 'features' / 'ds6'
REPORTS = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'stochastic_processes'
FIGURES = REPORTS / 'figures'
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

RNG_SEED        = 42
MIN_ACTIVE_DAYS = 7     # per period, per participant (reused from modeling_cautions.md S2)
MU_THRESHOLD    = 0.5   # attempts/day (FLAG 4 -- rescaled from DS5's 0.005 [0,1]-rate threshold)

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('ROOT    :', ROOT)
print('CLEANED :', CLEANED)
print('FEAT    :', FEAT)
print('REPORTS :', REPORTS)
print('FIGURES :', FIGURES)
print('Run at  :', MODEL_RUN_TS)

In [ ]:
# =============================================================================
# Load behavioral data
# =============================================================================
# NOTE: study_condition and study_period are BOTH native columns in
# ds6_behavioral_clean.csv -- no join required (unlike DS5, where study_condition
# must be joined from the survey file -- modeling_cautions.md S6).

beh = pd.read_csv(CLEANED / 'ds6_behavioral_clean.csv', low_memory=False)
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)
beh['event_date'] = beh['event_timestamp_utc'].dt.normalize()

print(f'Behavioral rows : {len(beh):,}')
print(f'Participants    : {beh["participant_id_canonical"].nunique():,}')
print(f'\nstudy_period value counts:')
print(beh['study_period'].value_counts())
assert set(beh['study_period'].unique()) == {'baseline', 'intervention'}, \
    'Unexpected study_period values -- blueprint assumption violated'
print('\nConfirmed: exactly 2 study_period values, as assumed in the blueprint.')

print(f'\nstudy_condition value counts:')
print(beh['study_condition'].value_counts())

print(f'\nevent_outcome_simple value counts (confirms single-outcome structure):')
print(beh['event_outcome_simple'].value_counts())

daily_events = beh.groupby(['participant_id_canonical', 'event_date']).size()
print(f'\nEvents per participant-day (all periods pooled):')
print(daily_events.describe().round(1))

In [ ]:
# =============================================================================
# Build zero-filled daily count grids, one per participant x period
# =============================================================================
# A day with zero attempts within a participant's active window is a real,
# meaningful zero -- not missing data (blueprint Part 2).

def build_zero_filled_grid(sub_df):
    """Given events for one participant x period, return a zero-filled daily Series
    indexed by calendar day from first to last active day (inclusive)."""
    if len(sub_df) == 0:
        return None
    counts = sub_df.groupby('event_date').size()
    full_range = pd.date_range(counts.index.min(), counts.index.max(), freq='D')
    grid = counts.reindex(full_range, fill_value=0)
    return grid

grid_records = []   # for daily_series_long.csv
param_input  = {}   # pid -> {period: grid}

for pid, pgrp in beh.groupby('participant_id_canonical'):
    param_input[pid] = {}
    for period in ['baseline', 'intervention']:
        sub = pgrp[pgrp['study_period'] == period]
        grid = build_zero_filled_grid(sub)
        param_input[pid][period] = grid
        if grid is not None:
            for day_idx, (date, n) in enumerate(grid.items()):
                grid_records.append(dict(
                    participant_id_canonical=pid, study_period=period,
                    event_date=date, day_index=day_idx, N_t=int(n),
                ))

daily_series_long = pd.DataFrame(grid_records)
print(f'daily_series_long rows: {len(daily_series_long):,}')
print(daily_series_long.groupby('study_period')['participant_id_canonical'].nunique())

In [ ]:
# =============================================================================
# Eligibility: MIN_ACTIVE_DAYS = 7, applied independently per period
# =============================================================================
# Reused from the already-verified modeling_cautions.md S2 finding (not re-derived).

active_days = (
    daily_series_long[daily_series_long['N_t'] > 0]
    .groupby(['participant_id_canonical', 'study_period'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['baseline', 'intervention'], fill_value=0)
)

elig_baseline     = set(active_days.index[active_days['baseline'] >= MIN_ACTIVE_DAYS])
elig_intervention = set(active_days.index[active_days['intervention'] >= MIN_ACTIVE_DAYS])
elig_both         = elig_baseline & elig_intervention

n_total = beh['participant_id_canonical'].nunique()
print(f'Total participants                 : {n_total}')
print(f'Baseline eligible (>={MIN_ACTIVE_DAYS} active days)     : {len(elig_baseline)} ({100*len(elig_baseline)/n_total:.1f}%)')
print(f'Intervention eligible (>={MIN_ACTIVE_DAYS} active days) : {len(elig_intervention)} ({100*len(elig_intervention)/n_total:.1f}%)')
print(f'Both periods eligible               : {len(elig_both)} ({100*len(elig_both)/n_total:.1f}%)')

---

## Section 3: Parameter Estimation

Estimating drift `mu_hat`, diffusion `sig_hat`, and burstiness `Fano_factor` per
participant x period (method of moments, FLAG 2).

In [ ]:
# =============================================================================
# SDE parameter estimation: method of moments (per participant x period)
# =============================================================================

def estimate_params(grid):
    """grid: zero-filled pandas Series of daily counts (chronological).
    Returns dict with mu_hat, sig_hat, fano, n_active_days, span_days, mean_N."""
    if grid is None or len(grid) < 2:
        return dict(mu_hat=np.nan, sig_hat=np.nan, fano=np.nan,
                    n_active_days=np.nan, span_days=np.nan, mean_N=np.nan)
    x = grid.values.astype(float)
    dx = np.diff(x)
    mu_hat  = float(np.mean(dx))
    sig_hat = float(np.std(dx, ddof=1)) if len(dx) >= 2 else np.nan
    mean_N  = float(np.mean(x))
    var_N   = float(np.var(x, ddof=1)) if len(x) >= 2 else np.nan
    fano    = var_N / mean_N if mean_N > 0 else np.nan
    return dict(mu_hat=mu_hat, sig_hat=sig_hat, fano=fano,
                n_active_days=int((x > 0).sum()), span_days=len(x), mean_N=mean_N)

records = []
for pid, periods in param_input.items():
    row = dict(participant_id_canonical=pid)
    for period in ['baseline', 'intervention']:
        grid = periods.get(period)
        elig = pid in (elig_baseline if period == 'baseline' else elig_intervention)
        if elig:
            p = estimate_params(grid)
        else:
            p = dict(mu_hat=np.nan, sig_hat=np.nan, fano=np.nan,
                      n_active_days=(int((grid.values > 0).sum()) if grid is not None else 0),
                      span_days=(len(grid) if grid is not None else 0), mean_N=np.nan)
        for k, v in p.items():
            row[f'{k}_{period}'] = v
    records.append(row)

params = pd.DataFrame(records)
print(f'Parameter records : {len(params):,}')
print(f'\nNaN counts:')
print(params[['mu_hat_baseline','sig_hat_baseline','mu_hat_intervention','sig_hat_intervention']].isna().sum())
print(f'\nSummary (baseline):')
print(params[['mu_hat_baseline','sig_hat_baseline','fano_baseline']].describe().round(3))
print(f'\nSummary (intervention):')
print(params[['mu_hat_intervention','sig_hat_intervention','fano_intervention']].describe().round(3))

In [ ]:
# =============================================================================
# Sanity check: mean_N vs events_per_day_baseline/intervention from Stage 1 features
# =============================================================================
# Small differences expected (zero-filled-grid mean vs Stage 1's active-day-only mean).
# Differences > 20% of the Stage-1 value warrant investigation.

beh_feat = pd.read_csv(FEAT / 'ds6_behavior_features.csv')
check = params.merge(
    beh_feat[['participant_id_canonical', 'events_per_day_baseline', 'events_per_day_intervention']],
    on='participant_id_canonical', how='left'
)
for period, feat_col in [('baseline', 'events_per_day_baseline'), ('intervention', 'events_per_day_intervention')]:
    mean_col = f'mean_N_{period}'
    diff = (check[mean_col] - check[feat_col]).abs()
    rel  = diff / check[feat_col].replace(0, np.nan)
    print(f'{period}: |mean_N - {feat_col}| describe:')
    print(diff.describe().round(3))
    print(f'  participants with relative diff > 20%: {(rel > 0.20).sum()}')
    print()

In [ ]:
# =============================================================================
# sigma/|mu| impulsivity ratio + h_period (FLAG 3)
# =============================================================================

for period in ['baseline', 'intervention']:
    mu_col, sig_col, ratio_col = f'mu_hat_{period}', f'sig_hat_{period}', f'sigma_mu_ratio_{period}'
    params[ratio_col] = np.where(
        params[mu_col].abs() < MU_THRESHOLD, np.nan, params[sig_col] / params[mu_col].abs()
    )

params['h_period'] = params['mu_hat_intervention'] - params['mu_hat_baseline']

n_nan_b = params['sigma_mu_ratio_baseline'].isna().sum()
n_nan_i = params['sigma_mu_ratio_intervention'].isna().sum()
print(f'sigma/|mu| ratio NaN (|mu| < {MU_THRESHOLD}): baseline={n_nan_b}, intervention={n_nan_i}')
print(f'\nh_period describe:')
print(params['h_period'].describe().round(3))
print(f'\nh_period > 0 (accelerating post-baseline): {(params["h_period"] > 0).sum()} / {params["h_period"].notna().sum()}')

---

## Section 4: Condition x Period Comparison

In [ ]:
# =============================================================================
# Join study_condition (native column -- no survey join needed, unlike DS5)
# =============================================================================

cond_map = beh.groupby('participant_id_canonical')['study_condition'].agg(
    lambda s: s.mode().iloc[0] if len(s) > 0 else np.nan
)
params['study_condition'] = params['participant_id_canonical'].map(cond_map)
print(params['study_condition'].value_counts())

# NOTE (modeling_cautions.md S6): study_condition = 'control' includes rows from BOTH
# periods -- the condition label does not itself change what study_period means.
# Both factors are reported jointly below, never collapsed into one.

cond_period_rows = []
for cond in ['control', 'default', 'intervention']:
    for period in ['baseline', 'intervention']:
        sub = params[params['study_condition'] == cond]
        mu  = sub[f'mu_hat_{period}'].dropna()
        sig = sub[f'sig_hat_{period}'].dropna()
        fano = sub[f'fano_{period}'].dropna()
        cond_period_rows.append(dict(
            study_condition=cond, study_period=period, n=len(mu),
            mu_mean=mu.mean(), mu_median=mu.median(), mu_std=mu.std(),
            sig_mean=sig.mean(), sig_median=sig.median(), sig_std=sig.std(),
            fano_mean=fano.mean(), fano_median=fano.median(),
        ))

condition_period_summary = pd.DataFrame(cond_period_rows).round(4)
print(condition_period_summary.to_string(index=False))

In [ ]:
# =============================================================================
# Inferential tests (preliminary, not confirmatory)
# =============================================================================

print('--- h_period by condition (paired within-participant drift shift) ---')
for cond in ['control', 'default', 'intervention']:
    vals = params.loc[params['study_condition'] == cond, 'h_period'].dropna()
    if len(vals) > 1:
        t_stat, p_val = stats.ttest_1samp(vals, 0)
        print(f'{cond:12s} n={len(vals):3d}  mean h_period={vals.mean():+.3f}  '
              f'(one-sample t vs 0: t={t_stat:.3f}, p={p_val:.4f})')

print('\n--- ANOVA: h_period across conditions ---')
groups = [params.loc[params['study_condition']==c, 'h_period'].dropna().values
          for c in ['control','default','intervention']]
groups = [g for g in groups if len(g) > 0]
if len(groups) >= 2:
    f_stat, p_anova = stats.f_oneway(*groups)
    print(f'ANOVA h_period ~ condition: F={f_stat:.3f}, p={p_anova:.4f}')

print('\n--- Paired test: mu_hat_baseline vs mu_hat_intervention (both-eligible participants) ---')
both = params.dropna(subset=['mu_hat_baseline', 'mu_hat_intervention'])
t_stat, p_val = stats.ttest_rel(both['mu_hat_baseline'], both['mu_hat_intervention'])
print(f'n={len(both)}  mean baseline mu={both["mu_hat_baseline"].mean():.3f}  '
      f'mean intervention mu={both["mu_hat_intervention"].mean():.3f}')
print(f'Paired t-test: t={t_stat:.3f}, p={p_val:.4f}')

In [ ]:
# =============================================================================
# Figure 1: mu_hat vs sig_hat scatter, colored by period
# =============================================================================
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=False, sharey=False)
for i, period in enumerate(['baseline', 'intervention']):
    sub = params.dropna(subset=[f'mu_hat_{period}', f'sig_hat_{period}'])
    ax[i].scatter(sub[f'mu_hat_{period}'], sub[f'sig_hat_{period}'], alpha=0.5, s=20)
    ax[i].axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax[i].set_xlabel('mu_hat (drift, attempts/day trend)')
    ax[i].set_ylabel('sig_hat (diffusion)')
    ax[i].set_title(f'DS6 StoP -- {period} period\n(state variable: daily attempt count, a behavioral proxy)')
plt.tight_layout()
fig.savefig(FIGURES / 'fig1_stop_mu_sigma_by_period.png', dpi=150)
plt.show()
print('Saved fig1_stop_mu_sigma_by_period.png')

In [ ]:
# =============================================================================
# Figure 2: h_period distribution
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 5))
vals = params['h_period'].dropna()
ax.hist(vals, bins=40, color='#4C72B0', alpha=0.8)
ax.axvline(0, color='black', linewidth=1)
ax.axvline(vals.mean(), color='red', linewidth=1, linestyle='--', label=f'mean={vals.mean():.3f}')
ax.set_xlabel('h_period = mu_hat_intervention - mu_hat_baseline')
ax.set_ylabel('Participants')
ax.set_title('DS6 StoP -- within-participant drift shift, baseline to intervention\n(behavioral proxy, not a causal claim without further design controls)')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig2_stop_h_period_distribution.png', dpi=150)
plt.show()
print('Saved fig2_stop_h_period_distribution.png')

In [ ]:
# =============================================================================
# Figure 3: Fano factor by period
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 5))
data_to_plot = [params['fano_baseline'].dropna(), params['fano_intervention'].dropna()]
bp = ax.violinplot(data_to_plot, showmeans=True)
ax.axhline(1.0, color='gray', linewidth=1, linestyle='--', label='Fano=1 (Poisson-like)')
ax.set_xticks([1, 2])
ax.set_xticklabels(['baseline', 'intervention'])
ax.set_ylabel('Fano factor = Var(N_t) / Mean(N_t)')
ax.set_title('DS6 StoP -- daily attempt-count burstiness by period')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig3_stop_fano_by_period.png', dpi=150)
plt.show()
print('Saved fig3_stop_fano_by_period.png')
print(f'\nMedian Fano: baseline={params["fano_baseline"].median():.2f}, intervention={params["fano_intervention"].median():.2f}')
print(f'Share Fano>1: baseline={100*(params["fano_baseline"]>1).mean():.1f}%, intervention={100*(params["fano_intervention"]>1).mean():.1f}%')

In [ ]:
# =============================================================================
# Figure 4: condition x period comparison (mu_hat)
# =============================================================================
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
conds = ['control', 'default', 'intervention']
x = np.arange(len(conds))
base_means = [condition_period_summary.query("study_condition==@c and study_period=='baseline'")['mu_mean'].values[0] for c in conds]
intv_means = [condition_period_summary.query("study_condition==@c and study_period=='intervention'")['mu_mean'].values[0] for c in conds]
ax.bar(x - width/2, base_means, width, label='baseline', color='#4C72B0')
ax.bar(x + width/2, intv_means, width, label='intervention', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(conds)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('mean mu_hat (drift)')
ax.set_title('DS6 StoP -- mean drift by study_condition x study_period')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig4_stop_condition_period_comparison.png', dpi=150)
plt.show()
print('Saved fig4_stop_condition_period_comparison.png')

---

## Section 5: Psychosocial Correlates

Preliminary, exploratory correlations between SDE parameters and DS6's wave-1 psychosocial
indices (`addiction_index_pre`, `self_control_index_pre` -- flagged in
`modeling_cautions.md` S4 as DS6's strongest self-control indicators).

In [ ]:
# =============================================================================
# Psychosocial correlations (exploratory)
# =============================================================================
survey_feat = pd.read_csv(FEAT / 'ds6_survey_features.csv')
merged = params.merge(
    survey_feat[['participant_id_canonical', 'addiction_index_pre', 'self_control_index_pre']],
    on='participant_id_canonical', how='left'
)

psych_cols = ['addiction_index_pre', 'self_control_index_pre']
sde_cols   = ['mu_hat_baseline', 'sig_hat_baseline', 'fano_baseline',
              'mu_hat_intervention', 'sig_hat_intervention', 'fano_intervention', 'h_period']

corr_records = []
for sde_col in sde_cols:
    for psych_col in psych_cols:
        sub = merged[[sde_col, psych_col]].dropna()
        if len(sub) >= 10:
            r, p = stats.pearsonr(sub[sde_col], sub[psych_col])
            corr_records.append(dict(sde_param=sde_col, psychosocial=psych_col,
                                      n=len(sub), r=round(r, 4), p=round(p, 4)))

psychosocial_correlations = pd.DataFrame(corr_records)
print(psychosocial_correlations.to_string(index=False))

In [ ]:
# =============================================================================
# Export all outputs
# =============================================================================
out_cols = ['participant_id_canonical', 'study_condition',
            'mu_hat_baseline', 'sig_hat_baseline', 'sigma_mu_ratio_baseline', 'fano_baseline',
            'n_active_days_baseline', 'span_days_baseline', 'mean_N_baseline',
            'mu_hat_intervention', 'sig_hat_intervention', 'sigma_mu_ratio_intervention', 'fano_intervention',
            'n_active_days_intervention', 'span_days_intervention', 'mean_N_intervention',
            'h_period']
participant_stop_parameters = params[out_cols].copy()
participant_stop_parameters.to_csv(REPORTS / 'participant_stop_parameters.csv', index=False)
print(f'Saved participant_stop_parameters.csv ({len(participant_stop_parameters)} rows, {len(out_cols)} cols)')

daily_series_long.to_csv(REPORTS / 'daily_series_long.csv', index=False)
print(f'Saved daily_series_long.csv ({len(daily_series_long)} rows)')

condition_period_summary.to_csv(REPORTS / 'condition_period_summary.csv', index=False)
print(f'Saved condition_period_summary.csv ({len(condition_period_summary)} rows)')

psychosocial_correlations.to_csv(REPORTS / 'psychosocial_correlations.csv', index=False)
print(f'Saved psychosocial_correlations.csv ({len(psychosocial_correlations)} rows)')

In [ ]:
# =============================================================================
# model_run_summary.md
# =============================================================================
n_both = participant_stop_parameters.dropna(subset=['mu_hat_baseline','mu_hat_intervention']).shape[0]
summary_md = f"""# DS6 StoP -- Model Run Summary

**Run at:** {MODEL_RUN_TS}
**Notebook:** notebooks/73_ds6_stochastic_processes.ipynb
**Blueprint:** outputs/reports/stage4/step1_modeling/ds6_stop_blueprint.md

## Eligibility
- Total participants: {n_total}
- Baseline eligible (>={MIN_ACTIVE_DAYS} active days): {len(elig_baseline)} ({100*len(elig_baseline)/n_total:.1f}%)
- Intervention eligible (>={MIN_ACTIVE_DAYS} active days): {len(elig_intervention)} ({100*len(elig_intervention)/n_total:.1f}%)
- Both periods eligible: {n_both} ({100*n_both/n_total:.1f}%)

## Key results
- Mean mu_hat (baseline): {params['mu_hat_baseline'].mean():.4f}
- Mean mu_hat (intervention): {params['mu_hat_intervention'].mean():.4f}
- Mean sig_hat (baseline): {params['sig_hat_baseline'].mean():.4f}
- Mean sig_hat (intervention): {params['sig_hat_intervention'].mean():.4f}
- Median Fano factor (baseline): {params['fano_baseline'].median():.3f}
- Median Fano factor (intervention): {params['fano_intervention'].median():.3f}
- Mean h_period (intervention - baseline drift): {params['h_period'].mean():+.4f}
- Paired t-test mu_hat_baseline vs mu_hat_intervention: t={t_stat:.3f}, p={p_val:.4f}

## Proxy warning
N_t is a count-based behavioral volume measurement (daily app-opening attempts). It does NOT
directly measure urge, self-control, habit strength, or any latent psychological state. All
downstream interpretation must maintain this distinction.

## Known limitations
- DS6 has no dismiss/close outcomes -- state variable is attempt volume, not override rate.
- study_condition is quasi-experimental (cohort-level), not individually randomized.
- A third study phase ("50% returned to baseline / 50% opened restrictions", per the raw
  project description) is not represented as a distinct study_period value in this export --
  flagged as an open unknown, not resolved here.
- h_period is a within-participant descriptive comparison, not a causal estimate.
"""

(REPORTS / 'model_run_summary.md').write_text(summary_md)
print(summary_md)

---

## Section 6: Interpretation and Limits

### Headline descriptive result

Mean daily attempt volume (`mean_N`) drops from **45.1 events/day (baseline)** to
**29.1 events/day (intervention)** -- roughly a one-third reduction, consistent in
direction and magnitude with the already-existing Stage 1
`event_rate_ratio_intervention_vs_baseline` feature (`data_docs/modeling_cautions.md` S4),
now independently cross-validated by this notebook's own zero-filled daily-grid
construction. This is a **level** effect (the resting daily volume changed).

The **trend** (drift, `h_period`) shows no corresponding systematic shift: mean h_period
= +0.081, paired t-test on mu_hat_baseline vs mu_hat_intervention t=-0.442, p=0.659 --
not significant. So while participants use their devices noticeably less during the
intervention period, their day-to-day *trajectory* within each period (accelerating vs
decelerating) does not detectably differ between periods. Level and trend are answering
different questions and should not be conflated -- this notebook shows one moved, the
other did not detectably move.

### Estimation-precision caution: window-length asymmetry (baseline vs intervention)

The baseline window is study-design-fixed at ~15 days (median span=15); intervention is
~28 days (median span=29). Baseline `sig_hat`/`Fano_factor` are therefore estimated from
roughly half as many daily increments as intervention's. The observed baseline > intervention
sig_hat/Fano gap (median Fano 6.58 vs 3.34) could reflect a genuine behavioral difference,
but a shorter window also mechanically produces a noisier (higher-variance) diffusion
estimate per participant, independent of any real change in day-to-day irregularity. This
notebook does not attempt to separate the two explanations -- flagged as an open caveat,
not resolved. Any downstream family (DyS, InT) building on these sig_hat/Fano values should
carry this caveat forward rather than treat the period contrast as a clean burstiness finding.

### What the StoP model reveals

| Result | Interpretation |
|---|---|
| mu_hat (baseline vs intervention) | Whether daily attempt-volume trend shifts once the intervention period begins |
| sig_hat, Fano_factor | Day-to-day irregularity / burstiness of attempt volume, separately per period |
| h_period | Within-participant drift shift -- DS6's own analog of DS5's condition effect, but a before/after comparison rather than a between-arm one |
| condition x period table | Whether the RCT-like arm (control/default/intervention) modulates the period effect |

### What this model does NOT show

- Whether attempts translate into problematic use, override, or any outcome-level construct
  (DS6 has no dismiss/close events to measure this against).
- A causal effect of the intervention -- `study_condition` is quasi-experimental (cohort-level
  assignment per `dataset_strengths_table.csv`), and the period comparison is naturally
  confounded with time (novelty effects, school calendar, seasonal use changes).
- Individual psychological states (urge, self-control) -- only their behavioral proxies.

### Forward connections (informational only -- not implemented here)

- **StoP -> InT (notebook 75):** entropy of the daily-count distribution per period,
  paralleling the DS5/Structured StoP-InT bridge.
- **StoP -> DyS (notebook 72):** low-sig_hat participants as candidates for a
  deterministic-regime characterization, re-derived for the count process (no [0,1] bound
  to exploit, unlike DS5's sigma->0 argument).
- **StoP -> Step 2 phenotyping:** (mu_hat, sig_hat, Fano_factor, h_period) per period are
  candidate phenotype dimensions, once DS6 Step 1 is further along.